# Task Arithmetic — Mask Calibration

## Preparing enviroment

In [1]:
# --- 0) Runtime check ---
import torch, sys, os
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.9.0+cu126
cuda available: True
gpu: Tesla T4


In [ ]:
# --- 1) Clone repo (task_arithmetic branch) ---
!git clone --single-branch https://github.com/caiopenayo/Federated-Learning-Under-the-Lens-of-Task-Arithmetic.git
%cd Federated-Learning-Under-the-Lens-of-Task-Arithmetic
!git pull origin task_arithmetic


# Make repo importable
repo_root = os.path.abspath(".")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

Cloning into 'Federated-Learning-Under-the-Lens-of-Task-Arithmetic'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 166 (delta 57), reused 110 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 242.07 KiB | 8.35 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/Federated-Learning-Under-the-Lens-of-Task-Arithmetic
From https://github.com/caiopenayo/Federated-Learning-Under-the-Lens-of-Task-Arithmetic
 * branch            task_arithmetic -> FETCH_HEAD
Already up to date.


## Loading essencials variables


In [5]:
# --- 2) Direct imports ---
from optim.fisher import (
    calibrate_gradient_mask_multi_round,
    MaskCalibrationConfig,
    compute_fisher_diag_scores
)

from models.vit_dino import build_dino_vit
from fl.dataloaders import build_federated_dataloaders
from data.datasets import get_cifar100, get_cifar100_transforms
from data.partition import make_dataset_loaders
from fl.fedavg import run_fedavg_experiment
from torch.utils.data import Subset, DataLoader
from google.colab import drive
drive.mount('/content/drive')

# --- 3) Build model ---
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_dino_vit(num_classes=100, img_size=160)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
print("model ready on", device)

# --- 4) Seting loader ---
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import numpy as np

transform_train, transform_test = get_cifar100_transforms(img_size=160)

train, val, test = get_cifar100(
    train_transform=transform_train,
    test_transform=transform_test,
    val_ratio=0.1,
    root="./data",
    seed=42
)

train_loader_centr, val_loader_centr, test_loader_centr = make_dataset_loaders(train, val, test)

calib_loader = DataLoader(val,   batch_size=8, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)



Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/86.7M [00:00<?, ?B/s]

model ready on cuda


100%|██████████| 169M/169M [00:04<00:00, 38.5MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


## Fine-tuning with Fisher algorithm and mask




In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt

# ============================================================
# Mask experiments with a large base budget (BASE_ROUNDS=440)
# Strategy:
#   - Run only a minimal set of scenarios.
#   - Keep the mask sweep small (few tf and calib_rounds).
#   - Optionally do a short "screening" stage before full 440 rounds.
# ============================================================

# ------------------------
# Mask sweep config (small grid)
# ------------------------
trainable_fractions = [0.2]   # 5% and 10% trainable parameters, equivalent to sparse_ratio
calib_rounds_list   = [3]         # 1 and 3 calibration rounds

cfg_template = dict(
    fisher_batches_per_round=5,
    rule="least_sensitive",
)

# ------------------------
# FedAvg parameters
# ------------------------
K = 100
C = 0.1
LR = 0.01

# Base budget (you requested up to 440 rounds)
BASE_ROUNDS = 440
BASE_J = 4

# ------------------------
# Minimal scenarios (to save time)
# ------------------------
# Main: hard heterogeneous + drift
# Sanity: near-IID-ish case
scenarios = [
    {"name": "main_hard",   "Nc": 10,  "J": 4}
    ]

# ------------------------
# IMPORTANT: Mask calibration loader
# ------------------------


def compute_mask_stats(mask_dict):
    total, trainable = 0, 0
    for m in mask_dict.values():
        total += m.numel()
        trainable += int((m != 0).sum())
    ratio = trainable / max(1, total)
    return total, trainable, ratio

all_histories = {}

print("=== Starting MASKED experiments (BASE_ROUNDS=440) ===")

for tf in trainable_fractions:
    for cr in calib_rounds_list:
        # Build mask calibration config
        cfg = MaskCalibrationConfig(
            trainable_fraction=tf,
            rounds=cr,
            fisher_batches_per_round=cfg_template["fisher_batches_per_round"],
            rule=cfg_template["rule"],
        )

        print("\n" + "=" * 72)
        print(f"[Mask] trainable_fraction={tf:.2f} | calib_rounds={cr} | rule={cfg.rule}")
        print("=" * 72)

        # Calibrate mask on a fresh copy to avoid contaminating the global model
        mask = calibrate_gradient_mask_multi_round(
            model=copy.deepcopy(model),
            dataloader=calib_loader,
            criterion=criterion,
            device=device,
            cfg=cfg,
        )

        total, trainable, ratio = compute_mask_stats(mask)
        print(f"Mask stats: trainable params {trainable:,}/{total:,} ({ratio:.2%})")

        mask_key = f"mask_tf={tf}_cr={cr}"
        all_histories[mask_key] = {}

        for sc in scenarios:
            Nc = sc["Nc"]
            J  = sc["J"]

            # Scale rounds to keep total local steps comparable to the BASE setting
            # scaled_rounds * J ≈ BASE_ROUNDS * BASE_J
            scaled_rounds = int(BASE_ROUNDS * (BASE_J / J))
            scaled_rounds = max(1, scaled_rounds)

            print(f"\n--- Scenario: {sc['name']} | Non-IID Nc={Nc}, J={J}, Rounds={scaled_rounds} ---")
            print(f"Generating dataloaders for Nc={Nc}...")

            loaders_noniid, _, _, *_ = build_federated_dataloaders(
                train_transform=transform_train,
                test_transform=transform_test,
                K=K,
                sharding="non_iid",
                Nc=Nc,
                val_ratio=0.1,
                batch_size=64,
                seed=42
            )

            ckpt_dir = "/content/drive/MyDrive/fl_ckpts"
            os.makedirs(ckpt_dir, exist_ok=True)

            exp_name = f"mask_tf{tf}_cr{cr}_Nc{Nc}_J{J}_R{scaled_rounds}"
            ckpt_path = os.path.join(ckpt_dir, exp_name + ".pth")

            hist = run_fedavg_experiment(
                base_model=copy.deepcopy(model),
                client_loaders=loaders_noniid,
                test_loader=test_loader_centr,
                rounds=scaled_rounds,
                C=C,
                J=J,
                lr=0.01,
                device=device,
                log_every=40,
                mask=mask,
                resume=True,          # <- if ckpt exists, resume; else start fresh
                ckpt_path=ckpt_path,
                ckpt_every=100,
            )


            all_histories[mask_key][sc["name"]] = hist

print("\n✓ Masked experiments complete")


=== Starting MASKED experiments (BASE_ROUNDS=440) ===

[Mask] trainable_fraction=0.20 | calib_rounds=3 | rule=least_sensitive
Mask stats: trainable params 4,333,460/21,667,300 (20.00%)

--- Scenario: main_hard | Non-IID Nc=10, J=4, Rounds=440 ---
Generating dataloaders for Nc=10...
  [FedAvg] Start: 440 rounds, C=0.1 (10 clients/round from 100 eligible), J=4 steps


## Graphics and analysis

In [ ]:
# ============================================================
# (Optional) Recover `all_histories` from checkpoints saved on Drive
# ============================================================
import os
import re
import glob
import torch

# Folder where this notebook writes checkpoints
CKPT_DIR = "/content/drive/MyDrive/fl_ckpts"  # adjust if needed

# If running on Colab and Drive is not mounted yet, mount it
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
except Exception:
    pass

# Example filenames produced above:
#   mask_tf0.2_cr3_Nc10_J4_R440.pth
fname_re = re.compile(
    r"mask_tf(?P<tf>[-0-9.eE]+)_cr(?P<cr>\d+)(?:_sc(?P<sc>[^_]+))?_Nc(?P<Nc>\d+)_J(?P<J>\d+)_R(?P<R>\d+)\.pth$"
)

ckpt_files = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pth")))
print(f"[load] Found {len(ckpt_files)} checkpoint(s) in: {CKPT_DIR}")

all_histories = {}

for ckpt_path in ckpt_files:
    fname = os.path.basename(ckpt_path)
    m = fname_re.match(fname)
    if not m:
        print(f"[load][skip] Unrecognized filename: {fname}")
        continue

    tf = str(float(m.group("tf")))
    cr = int(m.group("cr"))
    Nc = int(m.group("Nc"))
    J = int(m.group("J"))
    R = int(m.group("R"))

    scen_name = m.group("sc") or f"Nc={Nc}_J={J}_R={R}"
    mask_key = f"mask_tf={tf}_cr={cr}"

    ckpt = torch.load(ckpt_path, map_location="cpu")
    hist = ckpt.get("history", None)
    if hist is None:
        print(f"[load][skip] No 'history' in: {fname}")
        continue

    all_histories.setdefault(mask_key, {})[scen_name] = hist

print(f"[load] Loaded mask configs: {len(all_histories)}")
print(f"[load] Total histories: {sum(len(v) for v in all_histories.values())}")
print("[load] Keys:", list(all_histories.keys()))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _try_get_series(hist, keys):
    """Try to extract a 1D list/array from hist using candidate key paths."""
    if hist is None:
        return None

    # Case A: hist is dict with direct lists
    if isinstance(hist, dict):
        for k in keys:
            if k in hist and hist[k] is not None:
                v = hist[k]
                # flatten possible numpy/torch -> list
                try:
                    return list(v)
                except Exception:
                    pass

        # Case B: hist has nested dicts like hist['metrics']['test_acc']
        for kpath in keys:
            if isinstance(kpath, (tuple, list)):
                cur = hist
                ok = True
                for kk in kpath:
                    if isinstance(cur, dict) and kk in cur:
                        cur = cur[kk]
                    else:
                        ok = False
                        break
                if ok and cur is not None:
                    try:
                        return list(cur)
                    except Exception:
                        pass

    # Case C: hist is list of dicts, e.g. [{'round': 40, 'test_acc': ...}, ...]
    if isinstance(hist, list) and len(hist) > 0 and isinstance(hist[0], dict):
        for k in keys:
            if isinstance(k, str) and k in hist[0]:
                return [h.get(k, None) for h in hist]

    return None

def _infer_rounds(hist, acc):
    """Infer x-axis rounds from hist if possible, else use index."""
    # Try common patterns
    if isinstance(hist, dict):
        for k in ["rounds", "round", "r", "x_rounds"]:
            if k in hist and hist[k] is not None:
                xs = list(hist[k])
                if len(xs) == len(acc):
                    return xs

    if isinstance(hist, list) and len(hist) > 0 and isinstance(hist[0], dict):
        for k in ["round", "rnd", "r"]:
            if k in hist[0]:
                xs = [h.get(k, None) for h in hist]
                if len(xs) == len(acc) and all(x is not None for x in xs):
                    return xs

    # Fallback: 0..len-1
    return list(range(len(acc)))

def plot_all_test_accuracy(all_histories, *, title="FedAvg: Test Accuracy vs Round"):
    """
    Plots Test Acc curves for all (mask_key, scenario) entries in all_histories.
    Expects: all_histories[mask_key][scenario_name] = hist
    """
    plt.figure()
    plotted_any = False

    # Candidate keys for test accuracy
    acc_keys = [
        "test_acc", "test_accuracy", "acc_test", "accuracy_test",
        ("metrics", "test_acc"), ("metrics", "test_accuracy"),
        ("eval", "test_acc"), ("eval", "test_accuracy"),
    ]

    for mask_key, scen_dict in all_histories.items():
        for scen_name, hist in scen_dict.items():
            acc = _try_get_series(hist, acc_keys)
            if acc is None:
                print(f"[WARN] Could not find test accuracy in hist for {mask_key} / {scen_name}. Available keys: "
                      f"{list(hist.keys()) if isinstance(hist, dict) else type(hist)}")
                continue

            # Clean Nones
            acc_clean = [a for a in acc if a is not None]
            if len(acc_clean) == 0:
                print(f"[WARN] Empty test acc series for {mask_key} / {scen_name}")
                continue

            xs = _infer_rounds(hist, acc)
            # If we dropped Nones, align xs too (simple approach: keep only where acc != None)
            if any(a is None for a in acc):
                xs = [x for x, a in zip(xs, acc) if a is not None]
                acc = acc_clean

            # If accuracy is in [0,1], convert to %
            if max(acc) <= 1.0:
                acc = [100.0 * a for a in acc]

            label = f"{mask_key} | {scen_name}"
            plt.plot(xs, acc, label=label)
            plotted_any = True

    if not plotted_any:
        print("[ERROR] Nothing was plotted. Check the structure of all_histories / hist objects.")
        return

    plt.xlabel("Round")
    plt.ylabel("Test Accuracy (%)")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# --- Call it ---
plot_all_test_accuracy(all_histories)


In [ ]:
import inspect
print(inspect.signature(run_fedavg_experiment))
